In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
import linear_regression
import numpy as np

In [2]:
data = pd.read_csv('used_car_dataset.csv')

In [3]:
data.head()
## age, km, transmission, owner, fueltype

,Brand,model,Year,Age,kmDriven,Transmission,Owner,FuelType,PostedDate,AdditionInfo,AskPrice
0,Honda,City,2001,23,"98,000 km",Manual,second,Petrol,Nov-24,"Honda City v teck in mint condition, valid gen...","₹ 1,95,000"
1,Toyota,Innova,2009,15,190000.0 km,Manual,second,Diesel,Jul-24,"Toyota Innova 2.5 G (Diesel) 7 Seater, 2009, D...","₹ 3,75,000"
2,Volkswagen,VentoTest,2010,14,"77,246 km",Manual,first,Diesel,Nov-24,"Volkswagen Vento 2010-2013 Diesel Breeze, 2010...","₹ 1,84,999"
3,Maruti Suzuki,Swift,2017,7,"83,500 km",Manual,second,Diesel,Nov-24,Maruti Suzuki Swift 2017 Diesel Good Condition,"₹ 5,65,000"
4,Maruti Suzuki,Baleno,2019,5,"45,000 km",Automatic,first,Petrol,Nov-24,"Maruti Suzuki Baleno Alpha CVT, 2019, Petrol","₹ 6,85,000"


In [4]:
data = data.dropna()

In [5]:
data = pd.get_dummies(data, columns=['Transmission'], drop_first=True)

In [6]:
data = pd.get_dummies(data, columns=['Owner'], drop_first=True)

In [7]:
data = pd.get_dummies(data, columns=['FuelType'], drop_first=True)

In [8]:
data["kmDriven"] = (
    data["kmDriven"]
    .str.replace(",", "", regex=False)
    .str.replace(" km", "", regex=False)
    .astype(float)
)

## data['Car'] = data['Brand'].astype(str) + ' ' + data['model'].astype(str)

In [9]:
data["AskPrice"] = (
    data["AskPrice"]
    .str.replace(",", "", regex=False)
    .str.replace("₹ ", "", regex=False)
    .astype(float)
)

In [10]:
data.head()

,Brand,model,Year,Age,kmDriven,PostedDate,AdditionInfo,AskPrice,Transmission_Manual,Owner_second,FuelType_Hybrid/CNG,FuelType_Petrol
0,Honda,City,2001,23,98000.0,Nov-24,"Honda City v teck in mint condition, valid gen...",195000.0,1,1,0,1
1,Toyota,Innova,2009,15,190000.0,Jul-24,"Toyota Innova 2.5 G (Diesel) 7 Seater, 2009, D...",375000.0,1,1,0,0
2,Volkswagen,VentoTest,2010,14,77246.0,Nov-24,"Volkswagen Vento 2010-2013 Diesel Breeze, 2010...",184999.0,1,0,0,0
3,Maruti Suzuki,Swift,2017,7,83500.0,Nov-24,Maruti Suzuki Swift 2017 Diesel Good Condition,565000.0,1,1,0,0
4,Maruti Suzuki,Baleno,2019,5,45000.0,Nov-24,"Maruti Suzuki Baleno Alpha CVT, 2019, Petrol",685000.0,0,0,0,1


In [11]:
features = ['Age', 'kmDriven', 'Transmission_Manual', 'Owner_second', 'FuelType_Hybrid/CNG', 'FuelType_Petrol']

In [12]:
X = data[features]
y = data['AskPrice'].values

In [13]:
X.head()

,Age,kmDriven,Transmission_Manual,Owner_second,FuelType_Hybrid/CNG,FuelType_Petrol
0,23,98000.0,1,1,0,1
1,15,190000.0,1,1,0,0
2,14,77246.0,1,0,0,0
3,7,83500.0,1,1,0,0
4,5,45000.0,0,0,0,1


In [14]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [15]:
## Features to standardize: Age, kmDriven
scale_cols = ['Age', 'kmDriven']

# create transformer
preprocessor = ColumnTransformer(
    transformers=[
        ('scaler', StandardScaler(), scale_cols)
    ],
    remainder='passthrough'  # keeps other columns unchanged
)

X_train = preprocessor.fit_transform(X_train)

In [16]:
model = linear_regression.GradientDescentLR()
model.fit(X_train, y_train)

In [17]:
X_test = preprocessor.fit_transform(X_test)

In [18]:
print(model.get_theta())

[1711706.10173636 -431796.06629311 -120803.11369361 -794440.74449177
   65468.55712376 -703611.29129008 -380562.87468286]


In [19]:
pred = model.predict(X_test)
print(pred)

[1410747.59381096 1665319.78943435 1846270.62476922 ...  990688.80302169
 1720484.51879883 2035168.78457725]


In [20]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


mae = mean_absolute_error(y_test, pred)
mse = mean_squared_error(y_test, pred)
rmse = mean_squared_error(y_test, pred, squared=False)
r2 = r2_score(y_test, pred)

print(f"MAE: {mae}, RMSE: {rmse}, R²: {r2}")

MAE: 660812.2014279581, RMSE: 1337302.8978302665, R²: 0.21089762759375275


In [21]:
data.loc[2482]

Brand                                                         Land Rover
model                                                 Range Rover Evoque
Year                                                                2013
Age                                                                   11
kmDriven                                                         80500.0
PostedDate                                                        Oct-24
AdditionInfo           Land Rover Range Evoque Dynamic SD4, 2013, Diesel
AskPrice                                                       1590000.0
Transmission_Manual                                                    0
Owner_second                                                           1
FuelType_Hybrid/CNG                                                    0
FuelType_Petrol                                                        0
Name: 2482, dtype: object

In [25]:
model.predict(np.array([X_train[0]]))

array([1214531.4115508])